In [ ]:
"""
Fatigue measurement program
Applies continuous U-D stress cycling and takes 3PP measurements at defined time intervals
"""
import time
import numpy as np
from bnc765_driver import BNC765
from pulse_3pp import run_3pp


def setup_fatigue_pulser(pulser, amplitude, pulse_width_ns, spacing_ns,
                          base_offset=0.0, verbose=False):
    """
    Setup pulser in continuous U-D mode for fatigue cycling
    CH1: U pulses (positive), CH2: D pulses (negative)

    Args:
        pulser: BNC765 instance
        amplitude: Pulse amplitude (V)
        pulse_width_ns: Width of each pulse (ns)
        spacing_ns: Time between pulses (ns)
        base_offset: DC offset (V)
        verbose: Print info
    """
    pulser.stop()

    # Period = pulse width + spacing
    period_ns = pulse_width_ns + spacing_ns

    # CH1: U pulses (positive)
    pulser.ch1.output_state = False
    pulser.ch1.pulse_mode = 'SIN'
    pulser.ch1.inverted = False
    pulser.ch1.voltage_level = amplitude
    pulser.ch1.voltage_offset = base_offset + amplitude / 2
    pulser.ch1.load_impedance = 50
    pulser.ch1.period = f'{period_ns}ns'
    pulser.write(f'SOURCE1:PULSE1:WIDTH {pulse_width_ns}E-9')
    pulser.write(f'SOURCE1:PULSE1:DELAY 0')

    # CH2: D pulses (negative), offset by half period
    pulser.ch2.output_state = False
    pulser.ch2.pulse_mode = 'SIN'
    pulser.ch2.inverted = True
    pulser.ch2.voltage_level = amplitude
    pulser.ch2.voltage_offset = base_offset - amplitude / 2
    pulser.ch2.load_impedance = 50
    pulser.ch2.period = f'{period_ns}ns'
    pulser.write(f'SOURCE2:PULSE1:WIDTH {pulse_width_ns}E-9')
    pulser.write(f'SOURCE2:PULSE1:DELAY {pulse_width_ns + spacing_ns / 2}E-9')

    # Set to continuous mode [3]
    pulser.trigger_mode = 'CONTINUOUS'

    pulser.ch1.output_state = True
    pulser.ch2.output_state = True
    pulser.start()

    if verbose:
        frequency_hz = 1e9 / (2 * period_ns)
        print(f"Fatigue cycling configured:")
        print(f"  Amplitude: {amplitude} V")
        print(f"  Pulse width: {pulse_width_ns} ns")
        print(f"  Spacing: {spacing_ns} ns")
        print(f"  Frequency: {frequency_hz/1e6:.3f} MHz")

def generate_measurement_schedule(max_cycles, frequency_hz, max_time_s=None):
    """
    Generate measurement schedule:
    - At every decade of cycles (10, 100, 1000, ...)
    - Then every 300 seconds after the last decade

    Args:
        max_cycles: Maximum number of cycles to reach before switching to time-based
        frequency_hz: Cycling frequency (Hz) to convert cycles to time
        max_time_s: Maximum total test duration in seconds (None = run indefinitely)

    Returns:
        list of dicts: [{'type': 'cycles', 'value': N} or {'type': 'time', 'value': T}]
    """
    schedule = []

    # Decade-based measurements
    decade = 100
    while decade <= max_cycles:
        time_at_decade = decade / frequency_hz
        schedule.append({
            'type': 'cycles',
            'cycles': decade,
            'time_s': time_at_decade
        })
        decade *= 10

    # Time-based measurements every 300s after last decade
    last_decade_time = (decade / 10) / frequency_hz
    t = last_decade_time + 300

    while max_time_s is None or t <= max_time_s:
        schedule.append({
            'type': 'time',
            'cycles': t * frequency_hz,
            'time_s': t
        })
        t += 300

    return schedule


def run_fatigue(
    fatigue_amplitude,
    fatigue_pulse_width_ns,
    fatigue_spacing_ns,
    max_cycles,
    params_3pp,
    save_directory,
    base_offset=0.0,
    max_time_s=None,
    verbose=True,
    ):
    """
    Fatigue measurement with decade cycle-based then time-based 3PP characterization

    Measurement schedule:
    - Every decade: 10, 100, 1000, ... up to max_cycles
    - Every 300s thereafter until max_time_s
    """
    period_ns = fatigue_pulse_width_ns + fatigue_spacing_ns
    frequency_hz = 1e9 / period_ns

    # Generate measurement schedule
    schedule = generate_measurement_schedule(max_cycles, frequency_hz, max_time_s)

    if verbose:
        print(f"\n{'='*60}")
        print(f"FATIGUE MEASUREMENT")
        print(f"{'='*60}")
        print(f"Stress amplitude: {fatigue_amplitude} V")
        print(f"Stress frequency: {frequency_hz/1e6:.3f} MHz")
        print(f"Decade measurements up to: {max_cycles:.2e} cycles")
        print(f"Then every 300s")
        if max_time_s:
            print(f"Max duration: {max_time_s} s")
        print(f"Total measurement points: {len(schedule)}")
        print(f"{'='*60}\n")

    pulser = BNC765("TCPIP::169.254.125.69::INSTR")

    results = {
        'measurement_times_s': [],
        'cycles_at_measurement': [],
        'data': {'npp': [], 'pnn': []},
        'fatigue_params': {
            'amplitude': fatigue_amplitude,
            'pulse_width_ns': fatigue_pulse_width_ns,
            'spacing_ns': fatigue_spacing_ns,
            'frequency_hz': frequency_hz,
        }
    }

    try:
        # Initial 3PP measurement at t=0
        if verbose:
            print(f"\n{'='*40}")
            print(f"INITIAL 3PP MEASUREMENT (t=0, 0 cycles)")
            print(f"{'='*40}")

        cycle_dir = f"{save_directory}/t0s_c0"
        for polarity in ['npp', 'pnn']:
            data = run_3pp(
                **params_3pp,
                polarity=polarity,
                save_directory=cycle_dir,
                auto_trigger=True,
                save_data=True,
                save_plot=True,
                verbose=verbose
            )
            results['data'][polarity].append(data)

        results['measurement_times_s'].append(0)
        results['cycles_at_measurement'].append(0)

        # Start fatigue cycling
        if verbose:
            print(f"\nStarting fatigue cycling...")

        setup_fatigue_pulser(
            pulser=pulser,
            amplitude=fatigue_amplitude,
            pulse_width_ns=fatigue_pulse_width_ns,
            spacing_ns=fatigue_spacing_ns,
            base_offset=base_offset,
            verbose=verbose
        )

        test_start_time = time.perf_counter()

        for measurement in schedule:
            target_time_s = measurement['time_s']
            target_cycles = measurement['cycles']
            measurement_type = measurement['type']

            # Wait until target time
            if verbose:
                if measurement_type == 'cycles':
                    print(f"\nWaiting until {target_cycles:.2e} cycles "
                          f"(t={target_time_s:.2f}s)...")
                else:
                    print(f"\nWaiting until t={target_time_s:.1f}s "
                          f"({target_cycles:.2e} cycles)...")

            while True:
                elapsed = time.perf_counter() - test_start_time
                remaining = target_time_s - elapsed

                if remaining <= 0:
                    break

                if verbose and remaining > 10:
                    cycles_so_far = elapsed * frequency_hz
                    print(f"  t={elapsed:.1f}s / {target_time_s:.1f}s "
                          f"({cycles_so_far:.2e} cycles, "
                          f"{remaining:.1f}s remaining)")

                time.sleep(min(10.0, remaining))

            actual_elapsed = time.perf_counter() - test_start_time
            actual_cycles = actual_elapsed * frequency_hz

            # Stop fatigue cycling for measurement
            pulser.ch1.output_state = False
            pulser.ch2.output_state = False
            pulser.stop()
            time.sleep(0.5)

            if verbose:
                print(f"\n{'='*40}")
                print(f"3PP MEASUREMENT at t={actual_elapsed:.1f}s "
                      f"({actual_cycles:.2e} cycles)")
                print(f"{'='*40}")

            # Save directory named by both time and cycles
            cycle_dir = f"{save_directory}/t{int(target_time_s)}s_c{target_cycles:.2e}"

            for polarity in ['npp', 'pnn']:
                data = run_3pp(
                    **params_3pp,
                    polarity=polarity,
                    save_directory=cycle_dir,
                    auto_trigger=True,
                    save_data=True,
                    save_plot=True,
                    verbose=verbose,
                    extra_metadata={
                        'cycle_count': target_cycles,
                        'elapsed_time_s': actual_elapsed,
                        'fatigue_amplitude': fatigue_amplitude,
                        'fatigue_frequency_hz': frequency_hz
                    }
                )

                results['data'][polarity].append(data)

            results['measurement_times_s'].append(actual_elapsed)
            results['cycles_at_measurement'].append(actual_cycles)

            # Resume fatigue cycling
            if verbose:
                print(f"\nResuming fatigue cycling...")

            setup_fatigue_pulser(
                pulser=pulser,
                amplitude=fatigue_amplitude,
                pulse_width_ns=fatigue_pulse_width_ns,
                spacing_ns=fatigue_spacing_ns,
                base_offset=base_offset,
                verbose=False
            )

        if verbose:
            print(f"\n{'='*60}")
            print(f"FATIGUE MEASUREMENT COMPLETE")
            total_time = time.perf_counter() - test_start_time
            total_cycles = total_time * frequency_hz
            print(f"Total time: {total_time:.1f}s")
            print(f"Total cycles: {total_cycles:.2e}")
            print(f"Measurements taken: {len(results['measurement_times_s'])}")
            print(f"{'='*60}\n")

        return results

    except KeyboardInterrupt:
        print("\nFatigue measurement interrupted by user")
        return results

    finally:
        try:
            pulser.ch1.output_state = False
            pulser.ch2.output_state = False
            pulser.stop()
            pulser.shutdown()
        except:
            pass


# Example usage
if __name__ == "__main__":
    params_3pp = {
        'u_amplitude': 1.2,
        'u_to_n_delay': 500.0,
        'nd_amplitude': 1.2,
        'n_to_d_delay': 100.0,
        'base_offset': 0,
        'pulse_width_ns': 200.0,
        'capture_width_ns': 2000.0,
        'record_length': 10000,
        'vdiv': 0.02,
        'num_averages': 4,
        'save_plot': True,
        'save_data': True,
        'verbose': True,
    }

    results = run_fatigue(
        fatigue_amplitude=1.2,
        fatigue_pulse_width_ns=200.0,
        fatigue_spacing_ns=200.0,
        max_cycles=1e8,             # Decade measurements up to 1e8 cycles
        params_3pp=params_3pp,
        save_directory='79-1/ret2-cd20/fatigue',
        base_offset=0.0,
        max_time_s=7200,            # Then every 300s up to 2 hours
        verbose=True
    )